In [18]:
import duckdb
import json

print(duckdb.__version__)

1.5.6


In [4]:
conn = duckdb.connect("md:procurement")

In [7]:
conn.execute("""
    DESCRIBE current_procurements
""").df()

,column_name,column_type,null,key,default,extra
0,ocid,VARCHAR,YES,None,None,None
1,release_id,VARCHAR,YES,None,None,None
2,release_date,TIMESTAMP WITH TIME ZONE,YES,None,None,None
3,buyer_name,VARCHAR,YES,None,None,None
4,tender_title,VARCHAR,YES,None,None,None
5,tender_status,VARCHAR,YES,None,None,None
6,procurement_method,VARCHAR,YES,None,None,None
7,tender_value,"DECIMAL(19,2)",YES,None,None,None
8,currency,VARCHAR,YES,None,None,None


In [8]:
conn.execute("""
    DESCRIBE current_awards
""").df()

,column_name,column_type,null,key,default,extra
0,ocid,VARCHAR,YES,None,None,None
1,release_id,VARCHAR,YES,None,None,None
2,release_date,TIMESTAMP WITH TIME ZONE,YES,None,None,None
3,buyer_name,VARCHAR,YES,None,None,None
4,tender_title,VARCHAR,YES,None,None,None
5,award_id,VARCHAR,YES,None,None,None
6,award_title,VARCHAR,YES,None,None,None
7,award_status,VARCHAR,YES,None,None,None
8,award_date,TIMESTAMP WITH TIME ZONE,YES,None,None,None
9,award_value,"DECIMAL(19,2)",YES,None,None,None


In [11]:
conn.execute("""
    SELECT *
    FROM current_awards
    LIMIT 5
""").df()

C:\Users\rorym\AppData\Local\Temp\ipykernel_27212\3056035318.py:5: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  """).df()


,ocid,release_id,release_date,buyer_name,tender_title,award_id,award_title,award_status,award_date,award_value,award_value_gross,currency
0,ocds-h6vhtk-079029,101043-2026,2026-10-26 13:49:40+00:00,Leeds and York Partnership NHS Foundation Trust,PRO 032 2026_27 - Generator Hire,2,PRO 132 2026_27,active,2026-10-26 00:00:00+00:00,25000.00,30000.0,GBP
1,ocds-h6vhtk-06ee3d,101042-2026,2026-10-26 13:49:39+00:00,Thirteen Housing Group Limited,Health Surveillance Services,1,NaN,pending,2026-10-26 00:00:00+00:00,NaN,1416000.0,GBP
2,ocds-h6vhtk-06e715,101041-2026,2026-10-26 13:48:20+00:00,NaN,DLSITT1098 - Diamond-II Multipole Wiggler for K14,1,NaN,cancelled,NaT,NaN,NaN,NaN
3,ocds-h6vhtk-079031,101040-2026,2026-10-26 13:47:41+00:00,mhs homes group,Bottomline One PTX,1,NaN,active,NaT,NaN,NaN,NaN
4,ocds-h6vhtk-06e9be,101039-2026,2026-10-26 13:47:16+00:00,Department for Education,Cherry Tree Academy,1,NaN,active,2026-07-06 00:00:00+01:00,9732288.25,11678745.9,GBP


In [12]:
conn.execute("""
    SELECT
        ocid,
        COUNT(*) AS award_count
    FROM current_awards
    GROUP BY ocid
    HAVING COUNT(*) > 1
    ORDER BY award_count DESC
    LIMIT 20
""").df()

,ocid,award_count
0,ocds-h6vhtk-0789b4,107
1,ocds-h6vhtk-068056,85
2,ocds-h6vhtk-078de9,85
3,ocds-h6vhtk-077e14,65
4,ocds-h6vhtk-077e01,61
5,ocds-h6vhtk-077df6,58
6,ocds-h6vhtk-077df0,56
7,ocds-h6vhtk-077e10,52
8,ocds-h6vhtk-058142,45
9,ocds-h6vhtk-077e1a,40


In [14]:
conn.execute("""
    SELECT
        ocid,
        award_id,
        award_title,
        award_status,
        award_date,
        award_value,
        award_value_gross,
        currency
    FROM current_awards
    WHERE ocid = 'ocds-h6vhtk-0789b4'
    ORDER BY award_date, award_id
""").df()

C:\Users\rorym\AppData\Local\Temp\ipykernel_27212\2041218374.py:14: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  """).df()


,ocid,award_id,award_title,award_status,award_date,award_value,award_value_gross,currency
0,ocds-h6vhtk-0789b4,098041-2026-A0101-23,None,active,NaT,NaN,NaN,None
1,ocds-h6vhtk-0789b4,098041-2026-A0102-37,None,active,NaT,NaN,NaN,None
2,ocds-h6vhtk-0789b4,098041-2026-A0103-96,None,active,NaT,NaN,NaN,None
3,ocds-h6vhtk-0789b4,098041-2026-A0104-38,None,active,NaT,NaN,NaN,None
4,ocds-h6vhtk-0789b4,098041-2026-A0105-39,None,active,NaT,NaN,NaN,None
...,...,...,...,...,...,...,...,...
102,ocds-h6vhtk-0789b4,098041-2026-N147-6,None,active,NaT,NaN,NaN,None
103,ocds-h6vhtk-0789b4,098041-2026-N148-107,None,active,NaT,NaN,NaN,None
104,ocds-h6vhtk-0789b4,098041-2026-S608-7,None,active,NaT,NaN,NaN,None
105,ocds-h6vhtk-0789b4,098041-2026-S609-27,None,active,NaT,NaN,NaN,None


In [15]:
conn.execute("""
    SELECT
        COUNT(*) AS total_awards,
        COUNT(award_date) AS with_award_date,
        COUNT(award_value) AS with_award_value,
        COUNT(award_value_gross) AS with_gross_value,
        COUNT(currency) AS with_currency,
        COUNT(award_title) AS with_title
    FROM current_awards
""").df()

,total_awards,with_award_date,with_award_value,with_gross_value,with_currency,with_title
0,6768,1397,1317,1576,1576,1777


In [16]:
conn.execute("""
    DESCRIBE dim_suppliers
""").df()

,column_name,column_type,null,key,default,extra
0,supplier_key,VARCHAR,YES,None,None,None
1,supplier_id,VARCHAR,YES,None,None,None
2,supplier_name,VARCHAR,YES,None,None,None
3,release_id,VARCHAR,YES,None,None,None
4,release_date,TIMESTAMP WITH TIME ZONE,YES,None,None,None


In [17]:
conn.execute("""
    DESCRIBE award_suppliers
""").df()

,column_name,column_type,null,key,default,extra
0,ocid,VARCHAR,YES,None,None,None
1,release_id,VARCHAR,YES,None,None,None
2,award_id,VARCHAR,YES,None,None,None
3,supplier_key,VARCHAR,YES,None,None,None


In [19]:
row = conn.execute("""
    SELECT raw_json
    FROM raw_releases
    WHERE raw_json IS NOT NULL
    LIMIT 1
""").fetchone()

data = json.loads(row[0]) if isinstance(row[0], str) else row[0]

data.keys()

dict_keys(['id', 'tag', 'date', 'initiationType', 'ocid', 'parties', 'buyer', 'tender', 'awards', 'contracts', 'language'])

In [21]:
data["tender"].keys()

dict_keys(['id', 'legalBasis', 'title', 'description', 'status', 'procurementMethod', 'procurementMethodDetails', 'participationFees', 'techniques', 'lots', 'documents'])

In [23]:
data["tender"]["lots"][0].keys()

dict_keys(['id', 'status'])

In [24]:
data["parties"][0].keys()

dict_keys(['name', 'id', 'identifier', 'address', 'contactPoint', 'roles', 'details'])

In [25]:
data["parties"][0]["address"]

{'streetAddress': 'Great George Street',
 'locality': 'Leeds',
 'postalCode': 'LS1 3EX',
 'country': 'GB',
 'countryName': 'United Kingdom',
 'region': 'UKE42'}

In [27]:
data["parties"][0]["roles"]

['buyer']

In [28]:
conn.execute("""
    SELECT
        COUNT(*) AS buyer_parties,
        COUNT(p.value -> 'address' ->> 'locality') AS with_locality,
        COUNT(p.value -> 'address' ->> 'postalCode') AS with_postcode,
        COUNT(p.value -> 'address' ->> 'region') AS with_region,
        COUNT(p.value -> 'address' ->> 'country') AS with_country
    FROM raw_releases r,
         json_each(r.raw_json, '$.parties') AS p
    WHERE json_contains(p.value -> 'roles', '"buyer"')
""").df()

,buyer_parties,with_locality,with_postcode,with_region,with_country
0,9084,9084,9030,9083,7641


In [29]:
conn.execute("""
    SELECT
        r.release_id,
        COUNT(*) AS buyer_count
    FROM raw_releases r,
         json_each(r.raw_json, '$.parties') AS p
    WHERE json_contains(p.value -> 'roles', '"buyer"')
    GROUP BY r.release_id
    HAVING COUNT(*) > 1
    ORDER BY buyer_count DESC
    LIMIT 20
""").df()

,release_id,buyer_count
0,093955-2026,55
1,093101-2026,43
2,100490-2026,25
3,099981-2026,17
4,095084-2026,15
5,094212-2026,13
6,099603-2026,8
7,094083-2026,8
8,094963-2026,7
9,092808-2026,7


In [30]:
conn.execute("""
    SELECT
        p.value ->> 'name' AS buyer_name,
        p.value -> 'address' ->> 'locality' AS locality,
        p.value -> 'address' ->> 'postalCode' AS postcode,
        p.value -> 'address' ->> 'region' AS region
    FROM raw_releases r,
         json_each(r.raw_json, '$.parties') AS p
    WHERE r.release_id = '093955-2026'
      AND json_contains(p.value -> 'roles', '"buyer"')
""").df()

,buyer_name,locality,postcode,region
0,Scottish Police Authority,Glasgow,G40 4EH,UKM
1,West Lothian Council,Livingston,EH54 6FF,UKM78
2,West Dunbartonshire Council,Dumbarton,G82 1QL,UKM81
3,Stirling Council,Stirling,FK7 7QA,UKM77
4,South Lanarkshire Council,Hamilton,ML3 0AA,UKM95
5,South Ayrshire Council,Ayr,KA7 1DR,UKM94
6,Shetland Islands Council,"Lerwick, Shetland",ZE1 0LZ,UKM66
7,Scottish Borders Council,Newtown St Boswells,TD6 0SA,UKM91
8,Renfrewshire Council,Paisley,PA1 1JB,UKM83
9,Perth & Kinross Council,Perth,PH1 5PH,UKM77


In [34]:
conn.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(buyer_id) AS with_buyer_id,
        COUNT(*) - COUNT(buyer_id) AS without_buyer_id,
        COUNT(DISTINCT buyer_id) AS distinct_buyer_ids
    FROM stg_buyers
""").df()

,total_rows,with_buyer_id,without_buyer_id,distinct_buyer_ids
0,9084,9084,0,2108


In [36]:
conn.execute("""
    SELECT
        buyer_id,
        COUNT(*) AS appearances,
        COUNT(DISTINCT buyer_name) AS different_names,
        COUNT(DISTINCT postcode) AS different_postcodes
    FROM stg_buyers
    GROUP BY buyer_id
    HAVING COUNT(DISTINCT buyer_name) > 1
        OR COUNT(DISTINCT postcode) > 1
    ORDER BY appearances DESC
    LIMIT 20
""").df()

,buyer_id,appearances,different_names,different_postcodes
0,GB-PPON-PDQJ-7126-JDHG,195,3,1
1,GB-PPON-PHVX-4316-ZVGZ,73,1,14
2,GB-PPON-PJDG-6588-XDMM,69,2,1
3,GB-COH-02154540,64,1,2
4,GB-PPON-PNWW-1475-NYLN,63,2,1
5,GB-PPON-PVRL-5831-GLMM,52,4,3
6,GB-PPON-PBHM-7586-PWLP,40,1,2
7,GB-NHS-QHM,36,2,1
8,GB-PPON-PBZB-4962-TVLR,34,2,1
9,GB-PPON-PMPB-8772-VYWM,32,1,2


In [38]:
conn.execute("""
    SELECT
        COUNT(*) AS dimension_rows,
        COUNT(DISTINCT buyer_id) AS distinct_buyer_ids
    FROM dim_buyers
""").df()

,dimension_rows,distinct_buyer_ids
0,2108,2108


In [40]:
conn.execute("""
    SELECT
        COUNT(*) AS relationships,
        COUNT(DISTINCT ocid) AS procurements,
        COUNT(DISTINCT buyer_id) AS buyers
    FROM procurement_buyers
""").df()

,relationships,procurements,buyers
0,7914,7485,2108


In [42]:
conn.execute("""
    SELECT
        COUNT(*) AS total_procurements,
        COUNT(pb.ocid) AS with_buyer,
        COUNT(*) - COUNT(pb.ocid) AS without_buyer
    FROM current_procurements cp
    LEFT JOIN (
        SELECT DISTINCT ocid
        FROM procurement_buyers
    ) pb
        ON cp.ocid = pb.ocid
""").df()

,total_procurements,with_buyer,without_buyer
0,7485,7485,0


In [45]:
conn.execute("""
    SELECT
        COUNT(*) AS gold_rows,
        COUNT(DISTINCT ocid) AS distinct_procurements
    FROM gold_procurement_overview
""").df()

,gold_rows,distinct_procurements
0,7485,7485


In [47]:
conn.execute("""
    SELECT
        MIN(release_date) AS earliest_release,
        MAX(release_date) AS latest_release,
        COUNT(*) AS procurements
    FROM current_procurements
""").df()

C:\Users\rorym\AppData\Local\Temp\ipykernel_27212\69600666.py:7: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  """).df()


,earliest_release,latest_release,procurements
0,2026-09-30 14:05:26+01:00,2026-10-27 05:09:32+00:00,7485


In [49]:
conn.execute("""
    SELECT
        ocid,
        COUNT(*) AS release_count,
        MIN(release_date) AS first_release_date,
        MAX(release_date) AS latest_release_date
    FROM raw_releases
    GROUP BY ocid
    HAVING COUNT(*) > 1
    ORDER BY release_count DESC
    LIMIT 20
""").df()

C:\Users\rorym\AppData\Local\Temp\ipykernel_27212\1453125848.py:12: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  """).df()


,ocid,release_count,first_release_date,latest_release_date
0,ocds-h6vhtk-029897,23,2026-10-01 10:37:42+01:00,2026-10-22 14:48:28+01:00
1,ocds-h6vhtk-0510f8,18,2026-09-30 23:30:39+01:00,2026-10-14 23:30:45+01:00
2,ocds-h6vhtk-068647,13,2026-10-05 12:54:45+01:00,2026-10-05 12:55:43+01:00
3,ocds-h6vhtk-03dcd9,13,2026-10-12 16:20:52+01:00,2026-10-13 13:44:38+01:00
4,ocds-h6vhtk-078674,7,2026-10-13 12:37:51+01:00,2026-10-20 14:25:33+01:00
5,ocds-h6vhtk-078195,6,2026-10-06 12:27:48+01:00,2026-10-13 16:18:12+01:00
6,ocds-h6vhtk-0786a7,6,2026-10-13 14:28:03+01:00,2026-10-26 16:25:41+00:00
7,ocds-h6vhtk-029a8d,6,2026-10-05 11:20:00+01:00,2026-10-15 13:42:09+01:00
8,ocds-h6vhtk-078d2e,6,2026-10-21 15:45:11+01:00,2026-10-22 14:29:12+01:00
9,ocds-h6vhtk-06f69e,5,2026-10-01 08:48:27+01:00,2026-10-21 13:45:25+01:00


In [50]:
conn.execute("""
    WITH procurement_dates AS (
        SELECT
            ocid,
            MIN(release_date) AS first_release_date
        FROM raw_releases
        GROUP BY ocid
    )
    SELECT
        CAST(first_release_date AS DATE) AS procurement_date,
        COUNT(*) AS new_procurements
    FROM procurement_dates
    GROUP BY CAST(first_release_date AS DATE)
    ORDER BY procurement_date
""").df()

,procurement_date,new_procurements
0,2026-09-30,220
1,2026-10-01,412
2,2026-10-02,454
3,2026-10-04,6
4,2026-10-05,390
5,2026-10-06,403
6,2026-10-07,426
7,2026-10-08,397
8,2026-10-09,378
9,2026-10-10,2


In [51]:
conn.execute("""
    WITH first_releases AS (
        SELECT
            ocid,
            MIN(release_date) AS first_release_date
        FROM stg_releases
        GROUP BY ocid
    )

    SELECT
        CAST(r.release_date AS DATE) AS activity_date,
        COUNT(DISTINCT r.ocid) AS updated_procurements
    FROM stg_releases r
    INNER JOIN first_releases f
        ON r.ocid = f.ocid
    WHERE CAST(r.release_date AS DATE)
          > CAST(f.first_release_date AS DATE)
    GROUP BY CAST(r.release_date AS DATE)
    ORDER BY activity_date
""").df()

,activity_date,updated_procurements
0,2026-10-01,3
1,2026-10-02,13
2,2026-10-05,14
3,2026-10-06,16
4,2026-10-07,17
5,2026-10-08,30
6,2026-10-09,25
7,2026-10-10,1
8,2026-10-12,32
9,2026-10-13,32
